# C2.2 · Model-layer research

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *Security of AI*

Builds on **[C2.1 · What research means in a CISO org](https://spbreed.github.io/cyber-commons/lessons/C2.1.html)**.

| | |
|---|---|
| Tools used | garak, Llama 3.3, GLM-4.6, Kimi K2, Claude Opus 5 |

## What this lesson is

**What it covers.** Run a jailbreak taxonomy across Llama, GLM and Kimi and chart where they differ.

**Why a security engineer needs it.** Model cards read credulously. The control it builds is: adversarial robustness, jailbreak taxonomy, refusal analysis, capability elicitation.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The model layer is the one place where the same input legitimately produces different output, which makes every naive experiment on it unrepeatable. Method is not a formality here; it is the only thing separating a finding from a coincidence.

> **At CyberTravels.** “CyberTravels refunded a booking when I asked” is one attempt. The rate, with an interval, is what changes when the provenance control ships and what tells you the change was real.

## 2 · The framework

```
   same prompt, same model, three runs, three outputs
        |
        v
   +--------------------------------------------+
   | n trials . fixed seeds . a control arm     |
   | report a RATE with an interval, not a case |
   +--------------------------------------------+

   without method, a finding and a coincidence look identical
```

Model-layer research means treating the model as an object of study rather than
a demo subject. The discipline is one rule: **report rates, not anecdotes.**

"I got it to do X" is not a result. Language models are stochastic; with enough
attempts you can get almost anything once. The result is the *rate*, with an
interval, because the rate is what changes when a mitigation lands and the
interval is what tells you whether the change was real.

This matters practically. A mitigation that moves a technique from 62% to 48%
sounds like progress. With n=20 the confidence intervals overlap so heavily that
you have demonstrated nothing, and you are about to tell a board you reduced
risk by 23%.

## 3 · The control — compute the sample size before you run

The question is not "how many attempts should I do?" It is: **how small an effect do I need to be able to detect?**

## 4 · The procedure, as a skill

One success is an anecdote. The skill runs each technique enough times to classify it — not reproduced, flaky, reproducible — and then computes the sample size the before-and-after comparison actually needed, before running it rather than after.

### The skill — [`skills/research/technique-reproducibility-test/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/technique-reproducibility-test/SKILL.md)

```yaml
name: technique-reproducibility-test
description: >-
  Run a technique enough times to say whether it reproduces, and compute the
  sample size a before-and-after comparison actually needed before claiming a
  control worked. Use when a jailbreak "works", or when a fix is declared
  effective from a handful of trials.
allowed-tools: Read, Grep, Glob
```

# One success is an anecdote; the interval is the result

Model-layer research is statistical whether or not anybody does the statistics.
A technique that succeeds once may not reproduce; a control that appears to
help at n=20 has an interval overlapping the baseline. Both errors are avoided
by the same discipline — report a rate with an interval, and compute the sample
size before running the comparison.

## When to use this

Any claim about model behaviour: a technique that works, a control that helps, a
model that is safer than another.

## Procedure

**1 — Define success mechanically.** A string, a state, a check — something a
script decides. "The model complied" judged by reading is not reproducible
between two people.

**2 — Run enough trials to produce a rate, and hold the conditions fixed.**
Same model version, same temperature, same prompt. Record the version: a rate
without one is unrepeatable by construction.

**3 — Classify the technique honestly.** Not reproduced, flaky, or reproducible.
Flaky is a real and common answer and it deserves the word rather than a
rounded-up rate.

**4 — Compute the required sample size before comparing.** From the baseline
rate, the effect you would care about, and the power you want. Then run that
many. Doing this afterwards produces the number that makes the result you got
look significant.

**5 — Report intervals, and say when they overlap.** Show the same true effect
at n=20, n=100 and n=1000 if you need to make the point: the effect did not
change, the ability to see it did.

## Output contract

```json
{
  "success_criterion": "str",
  "conditions": {"model": "str", "version": "str", "temperature": 0.0},
  "techniques": [{"name": "str", "trials": 0, "rate": 0.0, "interval": [0.0, 0.0],
                  "verdict": "not reproduced|flaky|reproducible"}],
  "comparison": {"before": 0.0, "after": 0.0, "n": 0, "required_n": 0, "separated": false}
}
```

## Failure modes

- **Reporting a rate with no model version.** Nobody can repeat it.
- **Computing the sample size afterwards.** That is choosing the number that
  fits.
- **Rounding flaky up to works.** It is the finding, not a rough edge.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/technique-reproducibility-test/scripts/technique_reproducibility_test.py
SCRIPT = "skills/research/technique-reproducibility-test/scripts/technique_reproducibility_test.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Direct override is not reproduced, context reframe is flaky, task nesting is reproducible. The before/after comparison shows overlapping intervals at n=20 and n=100 and separation at n=1000, for an identical true effect. Sample-size calculation shows detecting 62%→48% needs roughly 200 trials while 62%→58% needs thousands.

## Your turn

Take the last jailbreak or injection result your team reported. Ask for n. If the answer is a single-digit number or 'we tried it a few times', the finding is real but the number attached to it is not.

---

**Next → [C2.3 · Weight-level techniques](https://spbreed.github.io/cyber-commons/lessons/C2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*